## Testar a conexão com o MongoDB
Confere se o Databricks consegue ler o Mongo do tracking mobile. Antes de rodar, o `.env` precisa ter `MONGO_URI` e, se o nome do banco não estiver na URI, `MONGO_DB`.

In [0]:
%pip install pymongo[srv] python-dotenv

In [0]:
dbutils.library.restartPython()

#### Conectar

In [0]:
import os
from dotenv import load_dotenv, find_dotenv
from pymongo import MongoClient

load_dotenv(find_dotenv(usecwd=True))

client = MongoClient(os.getenv("MONGO_URI"), serverSelectionTimeoutMS=8000)
db_name = os.getenv("MONGO_DB")
mongo_db = client[db_name] if db_name else client.get_default_database()

print(client.admin.command("ping"))

#### Listar as coleções e contar os documentos

In [0]:
for name in mongo_db.list_collection_names():
    print(name, mongo_db[name].count_documents({}))

#### Ver um documento de uma coleção
Troque `NOME_DA_COLECAO` por uma coleção da lista acima (a de tracking do mobile).

In [0]:
COLECAO = "tracking_events"
print(mongo_db[COLECAO].find_one())

#### Testar a conversão para Spark

In [0]:
import pandas as pd

def read_mongo(collection, limit=0):
    docs = list(mongo_db[collection].find({}, limit=limit))
    pdf = pd.json_normalize(docs)
    pdf = pdf.astype({c: "string" for c in pdf.columns if pdf[c].dtype == "object"})
    return spark.createDataFrame(pdf)

display(read_mongo(COLECAO, limit=20))